In [1]:
import keras
from keras.layers import Dense, Conv2D, BatchNormalization, Activation
from keras.layers import AveragePooling2D, Input, Flatten
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.regularizers import l2
from keras import backend as K
from keras.models import Model
from keras.datasets import cifar10
import numpy as np
import os


2025-08-24 04:42:43.942317: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-24 04:43:02.368930: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-24 04:43:16.563288: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
batch_size = 32
epochs = 10
data_augmentation = True
num_classes = 10
subtract_pixel_mean = True
n = 3
version = 1
if version == 1:
   depth = n * 6 + 2
elif version == 2:
   depth = n * 9 + 2
model_type = 'ResNet % dv % d' % (depth, version)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
input_shape = x_train.shape[1:]
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
if subtract_pixel_mean:
   x_train_mean = np.mean(x_train, axis=0)
   x_train -= x_train_mean
   x_test -= x_train_mean
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')
print('y_train shape:', y_train.shape)
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
x_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples
y_train shape: (50000, 1)


In [3]:
def lr_schedule(epoch):
   lr = 1e-3
   if epoch > 180:
       lr *= 0.5e-3
   elif epoch > 160:
       lr *= 1e-3
   elif epoch > 120:
       lr *= 1e-2
   elif epoch > 80:
       lr *= 1e-1
   print('Learning rate: ', lr)
   return lr

In [4]:
def resnet_layer(inputs,num_filters=16,kernel_size=3,strides=1,activation='relu',batch_normalization=True,conv_first=True):
   conv = Conv2D(num_filters,
                 kernel_size=kernel_size,
                 strides=strides,
                 padding='same',
                 kernel_initializer='he_normal',
                 kernel_regularizer=l2(1e-4))
   x = inputs
   if conv_first:
       x = conv(x)
       if batch_normalization:
           x = BatchNormalization()(x)
       if activation is not None:
           x = Activation(activation)(x)
   else:
       if batch_normalization:
           x = BatchNormalization()(x)
       if activation is not None:
           x = Activation(activation)(x)
       x = conv(x)
   return x

In [5]:
def resnet_v1(input_shape, depth, num_classes=10):
   if (depth - 2) % 6 != 0:
       raise ValueError('depth should be 6n + 2 (eg 20, 32, 44 in [a])')
   num_filters = 16
   num_res_blocks = int((depth - 2) / 6)
   inputs = Input(shape=input_shape)
   x = resnet_layer(inputs=inputs)
   for stack in range(3):
       for res_block in range(num_res_blocks):
           strides = 1
           if stack > 0 and res_block == 0:
               strides = 2
           y = resnet_layer(inputs=x, num_filters=num_filters, strides=strides)
           y = resnet_layer(inputs=y, num_filters=num_filters, activation=None)
           if stack > 0 and res_block == 0:
               x = resnet_layer(inputs=x, num_filters=num_filters, kernel_size=1, strides=strides, activation=None, batch_normalization=False)
           x = keras.layers.add([x, y])
           x = Activation('relu')(x)
       num_filters *= 2
   x = AveragePooling2D(pool_size=8)(x)
   y = Flatten()(x)
   outputs = Dense(num_classes, activation='softmax', kernel_initializer='he_normal')(y)
   model = Model(inputs=inputs, outputs=outputs)
   return model

In [6]:
def resnet_v2(input_shape, depth, num_classes=10):
   if (depth - 2) % 9 != 0:
       raise ValueError('depth should be 9n + 2 (eg 56 or 110 in [b])')
   num_filters_in = 16
   num_res_blocks = int((depth - 2) / 9)
   inputs = Input(shape=input_shape)
   x = resnet_layer(inputs=inputs,num_filters=num_filters_in,conv_first=True)
   for stage in range(3):
       for res_block in range(num_res_blocks):
           activation = 'relu'
           batch_normalization = True
           strides = 1
           if stage == 0:
               num_filters_out = num_filters_in * 4
               if res_block == 0:
                   activation = None
                   batch_normalization = False
           else:
               num_filters_out = num_filters_in * 2
               if res_block == 0:
                   strides = 2
           y = resnet_layer(inputs=x,num_filters=num_filters_in,kernel_size=1,strides=strides,activation=activation,batch_normalization=batch_normalization,conv_first=False)
           y = resnet_layer(inputs=y,num_filters=num_filters_in,conv_first=False)
           y = resnet_layer(inputs=y,num_filters=num_filters_out,kernel_size=1,conv_first=False)
           if res_block == 0:
               x = resnet_layer(inputs=x,num_filters=num_filters_out,kernel_size=1,strides=strides,activation=None,batch_normalization=False)
           x = keras.layers.add([x, y])
       num_filters_in = num_filters_out
   x = BatchNormalization()(x)
   x = Activation('relu')(x)
   x = AveragePooling2D(pool_size=8)(x)
   y = Flatten()(x)
   outputs = Dense(num_classes,activation='softmax',kernel_initializer='he_normal')(y)
   model = Model(inputs=inputs, outputs=outputs)
   return model

In [7]:
if version == 2:
   model = resnet_v2(input_shape=input_shape, depth=depth)
else:
   model = resnet_v1(input_shape=input_shape, depth=depth)
model.compile(loss='categorical_crossentropy',optimizer=Adam(learning_rate=lr_schedule(0)),metrics=['accuracy'])
model.summary()
print(model_type)
save_dir = os.path.join(os.getcwd(), 'saved_models')
model_name = 'cifar10_%s_model.{epoch:03d}.keras' % model_type
if not os.path.isdir(save_dir):
   os.makedirs(save_dir)
filepath = os.path.join(save_dir, model_name)
checkpoint = ModelCheckpoint(filepath=filepath,monitor='val_acc',verbose=1,save_best_only=True)
lr_scheduler = LearningRateScheduler(lr_schedule)
lr_reducer = ReduceLROnPlateau(factor=np.sqrt(0.1),cooldown=0,patience=5,min_lr=0.5e-6)
callbacks = [checkpoint, lr_reducer, lr_scheduler]
if not data_augmentation:
   print('Not using data augmentation.')
   model.fit(x_train, y_train,batch_size=batch_size,epochs=epochs,validation_data=(x_test, y_test),shuffle=True,callbacks=callbacks)
else:
   print('Using real-time data augmentation.')
   # Complete the ImageDataGenerator
   datagen = ImageDataGenerator(featurewise_center=False,samplewise_center=False,zca_whitening=False,rotation_range=20,width_shift_range=0.2,height_shift_range=0.2,horizontal_flip=True,fill_mode='nearest')
   # Fit the generator on the training data
   datagen.fit(x_train)
   # Use the generator for training
   model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),steps_per_epoch=x_train.shape[0] // batch_size,epochs=epochs,validation_data=(x_test, y_test),callbacks=callbacks)

2025-08-24 04:43:49.426843: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Learning rate:  0.001


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      2,320 │ activation[0][0]  │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │      2,320 │ activation_1[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ activation[0][0], │
│                     │ 16)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │      2,320 │ activation_2[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │      2,320 │ activation_3[0][… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 32,    │          0 │ activation_2[0][

 Total params: 274,442 (1.05 MB)

...

ResNet  20v  1
Using real-time data augmentation.


/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Learning rate:  0.001
Epoch 1/10
1562/1562 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step - accuracy: 0.3576 - loss: 1.9662

2025-08-24 04:49:58.212883: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 122880000 exceeds 10% of free system memory.


ValueError: ModelCheckpoint callback received monitor=val_acc, but Keras isn't able to automatically determine whether that metric should be maximized or minimized. Pass `mode='max'` in order to monitor based on the highest metric value, or pass `mode='min'` in order to use the lowest value.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt

# Evaluate on test set
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)
print(f'Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}')

# Predictions
y_prob = model.predict(x_test, batch_size=batch_size, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

# CIFAR-10 class names
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Classification report
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

# Helper to denormalize images for display
def _denorm(img):
    if subtract_pixel_mean:
        img = img + x_train_mean
    return np.clip(img, 0.0, 1.0)

# Show a grid of predictions
np.random.seed(42)
idx = np.random.choice(x_test.shape[0], 16, replace=False)
rows, cols = 4, 4
fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
for k, ax in enumerate(axes.ravel()):
    i = idx[k]
    img = _denorm(x_test[i])
    pred = y_pred[i]
    true = y_true[i]
    color = 'green' if pred == true else 'red'
    ax.imshow(img)
    ax.set_title(f'{class_names[pred]} ({class_names[true]})', color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()